In [12]:
import re

class Term:
    pass

class Variable(Term):
    def __init__(self, name):
        self.name = name
    def __repr__(self):
        return self.name

class Compound(Term):
    def __init__(self, op, args):
        self.op = op
        self.args = args
    def __repr__(self):
        if not self.args:
            return self.op
        return f"{self.op}({', '.join(map(str, self.args))})"

def parse_term(s):
    """Parses a string into Term objects using standard logic conventions."""
    s = s.replace(" ", "")
    tokens = re.split(r'([(),])', s)
    tokens = [t for t in tokens if t]

    def parse_expr(index):
        token = tokens[index]
        if index + 1 < len(tokens) and tokens[index + 1] == '(':
            op = token
            args = []
            index += 2
            while tokens[index] != ')':
                arg, index = parse_expr(index)
                args.append(arg)
                if tokens[index] == ',':
                    index += 1
            return Compound(op, args), index + 1
        else:
            if token.isupper() or token.lower() in ['u', 'v', 'w', 'x', 'y', 'z']:
                return Variable(token), index + 1
            else:
                return Compound(token, []), index + 1

    term, _ = parse_expr(0)
    return term

def substitute(term, subst):
    """Applies the substitution dictionary transitively down into a term."""
    if isinstance(term, Variable):
        if term.name in subst:
            return substitute(subst[term.name], subst)
        return term
    if isinstance(term, Compound):
        return Compound(term.op, [substitute(arg, subst) for arg in term.args])
    return term

def occurs_check(v, term, subst):
    """Returns True if variable v occurs anywhere inside term."""
    if v.name == getattr(term, 'name', None):
        return True
    if isinstance(term, Compound):
        return any(occurs_check(v, arg, subst) for arg in term.args)
    if isinstance(term, Variable) and term.name in subst:
        return occurs_check(v, subst[term.name], subst)
    return False

def unify(x, y, subst=None):
    if subst is None:
        subst = {}

    while isinstance(x, Variable) and x.name in subst:
        x = subst[x.name]
    while isinstance(y, Variable) and y.name in subst:
        y = subst[y.name]

    if repr(x) == repr(y):
        return subst

    if isinstance(x, Variable):
        if occurs_check(x, y, subst):
            return None
        # Add the new binding
        new_subst = {k: v for k, v in subst.items()}
        new_subst[x.name] = y
        # Transitively clean up and apply the new map to all existing substitutions
        for k in new_subst:
            new_subst[k] = substitute(new_subst[k], new_subst)
        return new_subst

    if isinstance(y, Variable):
        if occurs_check(y, x, subst):
            return None
        new_subst = {k: v for k, v in subst.items()}
        new_subst[y.name] = x
        # Transitively clean up and apply the new map to all existing substitutions
        for k in new_subst:
            new_subst[k] = substitute(new_subst[k], new_subst)
        return new_subst

    if isinstance(x, Compound) and isinstance(y, Compound):
        if x.op != y.op or len(x.args) != len(y.args):
            return None
        for arg_x, arg_y in zip(x.args, y.args):
            subst = unify(arg_x, arg_y, subst)
            if subst is None:
                return None
        return subst

    return None

def main():
    print("--- First-Order Logic Unifier (Fixed Chain Substitution) ---")
    print("Variables: x, y, z, w, u, v and ANY uppercase letter (e.g. X)")
    print("Constants: a, b, c, d, e...")
    print("-" * 60)

    try:
        str1 = input("Enter Expression 1: ")
        str2 = input("Enter Expression 2: ")

        term1 = parse_term(str1)
        term2 = parse_term(str2)

        result = unify(term1, term2)

        if result is None:
            print("\nUnification Failed!")
        else:
            print("\nUnification Successful!")
            print("Most General Unifier (MGU):")
            if not result:
                print("  { } (Expressions are already identical)")
            for var, mapping in sorted(result.items()):
                print(f"  {var} ➔ {mapping}")

            # Optional: Show what the unified final expression looks like
            print(f"\nUnified Expression Result:")
            print(f"  ➔ {substitute(term1, result)}")

    except Exception as e:
        print(f"\nParser Error: Check your brackets/commas. ({e})")

if __name__ == "__main__":
    main()


--- First-Order Logic Unifier (Fixed Chain Substitution) ---
Variables: x, y, z, w, u, v and ANY uppercase letter (e.g. X)
Constants: a, b, c, d, e...
------------------------------------------------------------
Enter Expression 1: Q(Y,G(A,B,B))
Enter Expression 2: Q(G(x,x),Y)

Unification Failed!
